In [190]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/moonknightmarvel/paysim/paysim.csv


In [191]:
import os
file_path = "/kaggle/input/datasets/moonknightmarvel/paysim"
print(os.listdir(file_path))

['paysim.csv']


In [192]:
import pandas as pd

file_path = "/kaggle/input/datasets/moonknightmarvel/paysim/paysim.csv"

df = pd.read_csv(file_path)

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [193]:
print(df.shape)

(6362620, 11)


In [194]:
print(df.columns)

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')


In [195]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [196]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [197]:
df['type'].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [198]:
df.groupby('type')['amount'].mean()

type
CASH_IN     168920.242004
CASH_OUT    176273.964346
DEBIT         5483.665314
PAYMENT      13057.604660
TRANSFER    910647.009645
Name: amount, dtype: float64

In [199]:
df.groupby('type')['amount'].median()

type
CASH_IN     143427.710
CASH_OUT    147072.185
DEBIT         3048.990
PAYMENT       9482.190
TRANSFER    486308.390
Name: amount, dtype: float64

In [200]:
df.groupby('type')['isFraud'].sum()

type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64

In [201]:
model_df = df.copy()

In [202]:
model_df.shape

(6362620, 11)

In [203]:
model_df['orig_balance_change'] = (model_df['oldbalanceOrg'] - model_df['newbalanceOrig'])

In [204]:
model_df['dest_balance_change'] = (model_df['newbalanceDest'] - model_df['oldbalanceDest'])

In [205]:
model_df[
[
    'amount',
     'oldbalanceOrg',
     'newbalanceOrig',
     'orig_balance_change',
     'oldbalanceDest',
     'newbalanceDest',
     'dest_balance_change'
]
].head(10)

,amount,oldbalanceOrg,newbalanceOrig,orig_balance_change,oldbalanceDest,newbalanceDest,dest_balance_change
0,9839.64,170136.00,160296.36,9839.64,0.0,0.00,0.00
1,1864.28,21249.00,19384.72,1864.28,0.0,0.00,0.00
2,181.00,181.00,0.00,181.00,0.0,0.00,0.00
3,181.00,181.00,0.00,181.00,21182.0,0.00,-21182.00
4,11668.14,41554.00,29885.86,11668.14,0.0,0.00,0.00
5,7817.71,53860.00,46042.29,7817.71,0.0,0.00,0.00
6,7107.77,183195.00,176087.23,7107.77,0.0,0.00,0.00
7,7861.64,176087.23,168225.59,7861.64,0.0,0.00,0.00
8,4024.36,2671.00,0.00,2671.00,0.0,0.00,0.00
9,5337.77,41720.00,36382.23,5337.77,41898.0,40348.79,-1549.21


In [206]:
model_df['orig_balance_discrepancy'] = abs(model_df['amount'] - model_df['orig_balance_change'])

In [207]:
model_df['dest_balance_discrepancy'] = abs(model_df['amount'] - model_df['dest_balance_change'])

In [208]:
model_df[
[
    'amount',
    'orig_balance_change',
     'orig_balance_discrepancy',
     'dest_balance_change',
     'dest_balance_discrepancy'
]
].head(10)

,amount,orig_balance_change,orig_balance_discrepancy,dest_balance_change,dest_balance_discrepancy
0,9839.64,9839.64,1.455192e-11,0.00,9839.64
1,1864.28,1864.28,1.136868e-12,0.00,1864.28
2,181.00,181.00,0.000000e+00,0.00,181.00
3,181.00,181.00,0.000000e+00,-21182.00,21363.00
4,11668.14,11668.14,0.000000e+00,0.00,11668.14
5,7817.71,7817.71,9.094947e-13,0.00,7817.71
6,7107.77,7107.77,1.091394e-11,0.00,7107.77
7,7861.64,7861.64,1.364242e-11,0.00,7861.64
8,4024.36,2671.00,1.353360e+03,0.00,4024.36
9,5337.77,5337.77,3.637979e-12,-1549.21,6886.98


### Temporal Frequency

In [209]:
model_df['transactions_per_step'] = (
    model_df.groupby(['nameOrig', 'step'])['amount']
            .transform('count')
)

The above code groups rows according to both the originating account AND the hour. 'transactions_per_step' therefore means the number of transactions made by a certain user in 1 hour. See how the output looks like below

In [210]:
model_df[
    ['nameOrig', 'step', 'amount', 'transactions_per_step']
].sort_values(
    by='transactions_per_step',
    ascending=False
).head(20)

,nameOrig,step,amount,transactions_per_step
1451734,C722055608,140,73597.75,2
1389388,C389626503,139,139133.13,2
4953465,C1194642233,351,10717.29,2
1422651,C1828219334,139,12788.63,2
371827,C884511024,17,288371.66,2
2459064,C1965131583,203,5961.67,2
4416739,C2084454803,322,83824.57,2
458746,C570148238,19,212959.86,2
5834340,C1061931265,402,31002.99,2
1714270,C407378873,160,8503.44,2


In [211]:
model_df['transactions_per_step'].value_counts().sort_index()

transactions_per_step
1    6362526
2         94
Name: count, dtype: int64

The above output means: 6,362,526 rows have transactions_per_step = 1, while only 94 rows have transactions_per_step = 2.

I was trying to use frequency of transactions to detect temporal frequency cz I thought that an unsual number of transactions by the same account within an hour can represent transaction-frequency anomalies but then as you have seen, there are no unusual frequency. The maximum is 2 so we'll have to use sth else.

In [212]:
model_df = model_df.sort_values('step').copy()

model_df['time_since_last_transaction'] = (
    model_df.groupby('nameOrig')['step'].diff()
)

In [213]:
model_df['time_since_last_transaction'].describe()

count    9313.000000
mean      158.312574
std       123.660909
min         0.000000
25%        56.000000
50%       139.000000
75%       225.000000
max       704.000000
Name: time_since_last_transaction, dtype: float64

In [214]:
model_df['time_since_last_transaction'].value_counts().sort_index().head(20)

time_since_last_transaction
0.0     47
1.0     65
2.0     68
3.0     57
4.0     77
5.0     60
6.0     47
7.0     39
8.0     28
9.0     37
10.0    23
11.0    12
12.0    17
13.0    21
14.0    33
15.0    27
16.0    32
17.0    42
18.0    55
19.0    47
Name: count, dtype: int64

The above result tells us something pretty important: time_since_last_transaction is also too sparse to be a strong feature for this PaySim dataset. So only about 0.15% of transactions have a usable time gap. The other ~99.85% are the first/only transaction from that nameOrig, so their value is NaN.

For those 9,313 repeated-account transactions, the median gap is 139 hours, the mean is about 158 hours, and gaps range from 0 to 704 hours. So the calculation works, but it doesn't give us enough coverage across the dataset. Therefore, we need another way to capture temporal frequency

****Another solution:
We can move from account-level frequency to system-level transaction frequency.

Instead of asking:
How frequently is this particular account transacting?

we can ask:
How much transaction activity is occurring during this hour?

In [215]:
transactions_per_hour = model_df.groupby('step').size()

transactions_per_hour.describe()

count      743.000000
mean      8563.418573
std      13388.693217
min          2.000000
25%         12.000000
50%        529.000000
75%      12317.000000
max      51352.000000
dtype: float64

Above we have grouped the transactions per hour and then used size() to count the number of rows per hour so in short we are checking the number transactions per hour

decribe() lets us inspect how hourly transaction volume varies across the whole simulation

In [216]:
transactions_per_hour.head(10)

step
1      2708
2      1014
3       552
4       565
5       665
6      1660
7      6837
8     21097
9     37628
10    35991
dtype: int64

The difference between this one and transaction per step that we started with above is that with transaction per step, we were looking at the number of transaction made by a particular person per hour. However with this one we are checking the number of transactions that happen per hour. We haven't also grouped using the user account.

In [217]:
model_df['hourly_transaction_volume'] = (
    model_df['step'].map(transactions_per_hour)
)

Above, 'hourly_transaction_volume' holds the transaction_per_hour value. The code maps every step(hour) to it's transaction/hour value and that value is held by 'hourly_transaction_volume' column

In [218]:
model_df[
    ['step', 'amount', 'hourly_transaction_volume']
].head(10)

,step,amount,hourly_transaction_volume
23,1,3876.41,2708
22,1,1065.41,2708
21,1,9302.79,2708
20,1,1373.43,2708
19,1,215310.30,2708
18,1,671.64,2708
17,1,1157.86,2708
16,1,1563.82,2708
15,1,229133.94,2708
14,1,4098.78,2708


In [219]:
hourly_volume_change = transactions_per_hour.diff()

hourly_volume_change.head(10)

step
1         NaN
2     -1694.0
3      -462.0
4        13.0
5       100.0
6       995.0
7      5177.0
8     14260.0
9     16531.0
10    -1637.0
dtype: float64

In [220]:
hourly_volume_change_abs = hourly_volume_change.abs()

model_df['hourly_volume_change'] = (
    model_df['step'].map(hourly_volume_change_abs)
)

In [221]:
model_df[
    ['step',
     'amount',
     'hourly_transaction_volume',
     'hourly_volume_change']
].drop_duplicates(subset='step').head(10)

,step,amount,hourly_transaction_volume,hourly_volume_change
23,1,3876.41,2708,NaN
2714,2,5427.59,1014,1694.0
3741,3,194242.46,552,462.0
4281,4,1036.33,565,13.0
4972,5,357885.34,665,100.0
5516,6,12848.12,1660,995.0
7296,7,41664.88,6837,5177.0
14121,8,229393.25,21097,14260.0
60269,9,58293.61,37628,16531.0
96756,10,7125.54,35991,1637.0


#### Transaction amount variance

In [222]:
amount_stats = model_df.groupby('type')['amount'].agg(
    ['mean', 'std', 'var']
)

amount_stats

,mean,std,var
type,,,
CASH_IN,168920.242004,1.265083e+05,1.600434e+10
CASH_OUT,176273.964346,1.753297e+05,3.074052e+10
DEBIT,5483.665314,1.331854e+04,1.773834e+08
PAYMENT,13057.604660,1.255645e+04,1.576644e+08
TRANSFER,910647.009645,1.879574e+06,3.532797e+12


In [223]:
type_mean_amount = model_df.groupby('type')['amount'].transform('mean')

The above code maps the mean amounts we've gotten to their respective rows. So if a row is a cash_in transaction, type_mean_amount will hold 168920 which is the mean of cash_in transactions

In [224]:
model_df['amount_deviation'] = abs(
    model_df['amount'] - type_mean_amount
)

Basically amount_deviation means: How far is my amount from the average amount(mean) of my transaction type?

In [225]:
model_df[
    ['type', 'amount', 'amount_deviation']
].sort_values(
    'amount_deviation',
    ascending=False
).head(10)

,type,amount,amount_deviation
3686583,TRANSFER,92445516.64,9.153487e+07
4060598,TRANSFER,73823490.36,7.291284e+07
4146397,TRANSFER,71172480.42,7.026183e+07
3946920,TRANSFER,69886731.30,6.897608e+07
3911956,TRANSFER,69337316.27,6.842667e+07
3937152,TRANSFER,67500761.29,6.659011e+07
4105338,TRANSFER,66761272.21,6.585063e+07
3892529,TRANSFER,64234448.19,6.332380e+07
3991638,TRANSFER,63847992.58,6.293735e+07
4143801,TRANSFER,63294839.63,6.238419e+07


In [226]:
type_mean = model_df.groupby('type')['amount'].transform('mean')
type_std = model_df.groupby('type')['amount'].transform('std')

model_df['amount_zscore'] = (
    (model_df['amount'] - type_mean) / type_std
)

In [227]:
model_df['amount_zscore_abs'] = model_df['amount_zscore'].abs()

In [228]:
model_df[
    ['type',
     'amount',
     'amount_deviation',
     'amount_zscore_abs']
].sort_values(
    'amount_zscore_abs',
    ascending=False
).head(10)

,type,amount,amount_deviation,amount_zscore_abs
6362579,CASH_OUT,10000000.0,9.823726e+06,56.030003
6362507,CASH_OUT,10000000.0,9.823726e+06,56.030003
5987864,CASH_OUT,10000000.0,9.823726e+06,56.030003
6266410,CASH_OUT,10000000.0,9.823726e+06,56.030003
6266412,CASH_OUT,10000000.0,9.823726e+06,56.030003
6266408,CASH_OUT,10000000.0,9.823726e+06,56.030003
6266406,CASH_OUT,10000000.0,9.823726e+06,56.030003
4154973,CASH_OUT,10000000.0,9.823726e+06,56.030003
6296669,CASH_OUT,10000000.0,9.823726e+06,56.030003
6296011,CASH_OUT,10000000.0,9.823726e+06,56.030003


In [229]:
model_df['amount_zscore_abs'].describe()

count    6.362620e+06
mean     6.676703e-01
std      7.444566e-01
min      1.328203e-07
25%      2.732351e-01
50%      5.434899e-01
75%      8.749816e-01
max      5.603000e+01
Name: amount_zscore_abs, dtype: float64

In [230]:
model_df.groupby('type')[
    ['dest_balance_change', 'dest_balance_discrepancy']
].agg(['mean', 'median'])

dest_balance_change             dest_balance_discrepancy  \
                        mean      median                     mean   
type                                                                
CASH_IN       -120813.411259 -104596.800            312986.774680   
CASH_OUT       193568.180775  150187.380             30283.193936   
DEBIT           19867.696753    3059.405             23945.596075   
PAYMENT             0.000000       0.000             13057.604660   
TRANSFER       986961.106232  493726.170            113966.567108   

                        
                median  
type                    
CASH_IN   2.543064e+05  
CASH_OUT  5.820766e-11  
DEBIT     5.127276e-11  
PAYMENT   9.482190e+03  
TRANSFER  1.164153e-10

In [231]:
model_df.groupby('type').apply(
    lambda x: (
        (x['oldbalanceDest'] == 0) &
        (x['newbalanceDest'] == 0)
    ).mean()
)

/tmp/ipykernel_58/2113499134.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  model_df.groupby('type').apply(


type
CASH_IN     0.114348
CASH_OUT    0.000719
DEBIT       0.000000
PAYMENT     1.000000
TRANSFER    0.007832
dtype: float64

In [232]:
numerical_features = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs'
]

In [233]:
model_df['hourly_volume_change'] = (
    model_df['hourly_volume_change'].fillna(0)
)

In [234]:
model_df[numerical_features].isnull().sum()

amount                      0
oldbalanceOrg               0
newbalanceOrig              0
orig_balance_change         0
orig_balance_discrepancy    0
amount_zscore_abs           0
dtype: int64

In [235]:
feature_columns = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs',
    'type'
]

X = model_df[feature_columns].copy()

In [236]:
X = pd.get_dummies(
    X,
    columns=['type'],
    dtype=int
)

In [237]:
X.shape

(6362620, 11)

In [238]:
X.columns.tolist()

['amount',
 'oldbalanceOrg',
 'newbalanceOrig',
 'orig_balance_change',
 'orig_balance_discrepancy',
 'amount_zscore_abs',
 'type_CASH_IN',
 'type_CASH_OUT',
 'type_DEBIT',
 'type_PAYMENT',
 'type_TRANSFER']

In [239]:
y = model_df['isFraud'].copy()

In [240]:
y.value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [241]:
model_df = model_df.sort_values(
    by=['step', 'nameOrig', 'nameDest'],
    kind='mergesort'
).reset_index(drop=True)

In [242]:
train_df = model_df[model_df['step'] <= 355].copy()
test_df = model_df[model_df['step'] > 355].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("Training steps:", train_df['step'].min(), "to", train_df['step'].max())
print("Testing steps:", test_df['step'].min(), "to", test_df['step'].max())

Training shape: (5113884, 22)
Testing shape: (1248736, 22)
Training steps: 1 to 355
Testing steps: 356 to 743


In [243]:
print("Training labels:")
print(train_df['isFraud'].value_counts())

print("\nTesting labels:")
print(test_df['isFraud'].value_counts())

Training labels:
isFraud
0    5109921
1       3963
Name: count, dtype: int64

Testing labels:
isFraud
0    1244486
1       4250
Name: count, dtype: int64


In [244]:
train_type_stats = train_df.groupby('type')['amount'].agg(
    mean='mean',
    std='std'
)

train_type_stats

,mean,std
type,,
CASH_IN,168764.154603,1.265988e+05
CASH_OUT,176473.498231,1.561471e+05
DEBIT,5592.754116,1.262620e+04
PAYMENT,12854.768219,1.239073e+04
TRANSFER,930792.038517,2.023719e+06


In [245]:
train_df['type_amount_mean'] = (
    train_df['type'].map(train_type_stats['mean'])
)

train_df['type_amount_std'] = (
    train_df['type'].map(train_type_stats['std'])
)

train_df['amount_zscore_abs'] = (
    (
        train_df['amount'] - train_df['type_amount_mean']
    ) / train_df['type_amount_std']
).abs()

In [246]:
test_df['type_amount_mean'] = (
    test_df['type'].map(train_type_stats['mean'])
)

test_df['type_amount_std'] = (
    test_df['type'].map(train_type_stats['std'])
)

test_df['amount_zscore_abs'] = (
    (
        test_df['amount'] - test_df['type_amount_mean']
    ) / test_df['type_amount_std']
).abs()

In [247]:
print("Training:")
print(train_df['amount_zscore_abs'].describe())

print("\nTesting:")
print(test_df['amount_zscore_abs'].describe())

print("\nMissing values:")
print("Train:", train_df['amount_zscore_abs'].isnull().sum())
print("Test:", test_df['amount_zscore_abs'].isnull().sum())

Training:
count    5.113884e+06
mean     6.909754e-01
std      7.228776e-01
min      1.437048e-07
25%      2.813004e-01
50%      5.679472e-01
75%      9.129855e-01
max      6.291200e+01
Name: amount_zscore_abs, dtype: float64

Testing:
count    1.248736e+06
mean     7.039966e-01
std      9.808635e-01
min      3.088822e-07
25%      2.836719e-01
50%      5.686142e-01
75%      9.175604e-01
max      6.291200e+01
Name: amount_zscore_abs, dtype: float64

Missing values:
Train: 0
Test: 0


In [248]:
numerical_features = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs'
]

X_train = train_df[numerical_features + ['type']].copy()
X_test = test_df[numerical_features + ['type']].copy()

In [249]:
X_train = pd.get_dummies(
    X_train,
    columns=['type'],
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=['type'],
    dtype=int
)

In [250]:
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [251]:
y_train = train_df['isFraud'].copy()
y_test = test_df['isFraud'].copy()

In [252]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

print("\nMissing values:")
print("Train:", X_train.isnull().sum().sum())
print("Test:", X_test.isnull().sum().sum())

X_train shape: (5113884, 11)
X_test shape: (1248736, 11)

Features:
['amount', 'oldbalanceOrg', 'newbalanceOrig', 'orig_balance_change', 'orig_balance_discrepancy', 'amount_zscore_abs', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']

Missing values:
Train: 0
Test: 0


In [253]:
from sklearn.ensemble import IsolationForest

In [254]:
iso_forest = IsolationForest(
    n_estimators=100,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)

In [255]:
iso_forest.fit(X_train)

IsolationForest(n_jobs=-1, random_state=42)

In [256]:
print("Isolation Forest training complete!")

Isolation Forest training complete!


In [257]:
test_scores = iso_forest.decision_function(X_test)

test_df['anomaly_score'] = test_scores

In [258]:
test_df['anomaly_score'].describe()

count    1.248736e+06
mean     7.126963e-02
std      6.453197e-02
min     -2.891450e-01
25%      3.734249e-02
50%      9.669826e-02
75%      1.210831e-01
max      1.358705e-01
Name: anomaly_score, dtype: float64

In [259]:
test_predictions = iso_forest.predict(X_test)

test_df['anomaly_prediction'] = test_predictions

In [260]:
test_df['anomaly_prediction'].value_counts()

anomaly_prediction
 1    1076575
-1     172161
Name: count, dtype: int64

In [261]:
anomaly_count = (
    test_df['anomaly_prediction'] == -1
).sum()

anomaly_percentage = (
    anomaly_count / len(test_df)
) * 100

print("Anomalies detected:", anomaly_count)
print("Percentage flagged:", anomaly_percentage)

Anomalies detected: 172161
Percentage flagged: 13.78682123363145


In [262]:
test_df['predicted_anomaly'] = (
    test_df['anomaly_prediction'] == -1
).astype(int)

In [263]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(
    test_df['isFraud'],
    test_df['predicted_anomaly']
)

print(cm)

[[1075364  169122]
 [   1211    3039]]


In [264]:
print(
    classification_report(
        test_df['isFraud'],
        test_df['predicted_anomaly'],
        digits=4
    )
)

              precision    recall  f1-score   support

           0     0.9989    0.8641    0.9266   1244486
           1     0.0177    0.7151    0.0345      4250

    accuracy                         0.8636   1248736
   macro avg     0.5083    0.7896    0.4805   1248736
weighted avg     0.9955    0.8636    0.9236   1248736



In [265]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [
    0.00,
    -0.02,
    -0.04,
    -0.06,
    -0.08,
    -0.10,
    -0.12,
    -0.14,
    -0.16,
    -0.18,
    -0.20
]

results = []

for threshold in thresholds:
    predictions = (
        test_df['anomaly_score'] < threshold
    ).astype(int)

    precision = precision_score(
        test_df['isFraud'],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        test_df['isFraud'],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        test_df['isFraud'],
        predictions,
        zero_division=0
    )

    flagged = predictions.sum()

    flagged_percentage = (
        flagged / len(predictions)
    ) * 100

    results.append({
        'threshold': threshold,
        'flagged': flagged,
        'flagged_percentage': flagged_percentage,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

threshold_results = pd.DataFrame(results)

threshold_results

,threshold,flagged,flagged_percentage,precision,recall,f1
0,0.00,172161,13.786821,0.017652,0.715059,0.034454
1,-0.02,124220,9.947659,0.022017,0.643529,0.042578
2,-0.04,83256,6.667222,0.031397,0.615059,0.059744
3,-0.06,58261,4.665598,0.039632,0.543294,0.073875
4,-0.08,41202,3.299496,0.050216,0.486824,0.091041
5,-0.10,30295,2.426053,0.062651,0.446588,0.109886
6,-0.12,22202,1.777958,0.071030,0.371059,0.119235
7,-0.14,15609,1.249984,0.077840,0.285882,0.122363
8,-0.16,10719,0.858388,0.090400,0.228000,0.129468
9,-0.18,5045,0.404009,0.167493,0.198824,0.181818


In [266]:
validation_df = test_df[
    test_df['step'] <= 549
].copy()

final_test_df = test_df[
    test_df['step'] >= 550
].copy()

In [267]:
print("Validation shape:", validation_df.shape)
print(
    "Validation steps:",
    validation_df['step'].min(),
    "to",
    validation_df['step'].max()
)

print("\nFinal test shape:", final_test_df.shape)
print(
    "Final test steps:",
    final_test_df['step'].min(),
    "to",
    final_test_df['step'].max()
)

print("\nValidation labels:")
print(validation_df['isFraud'].value_counts())

print("\nFinal test labels:")
print(final_test_df['isFraud'].value_counts())

Validation shape: (1047331, 27)
Validation steps: 356 to 549

Final test shape: (201405, 27)
Final test steps: 550 to 743

Validation labels:
isFraud
0    1045237
1       2094
Name: count, dtype: int64

Final test labels:
isFraud
0    199249
1      2156
Name: count, dtype: int64


In [268]:
X_validation = validation_df[
    numerical_features + ['type']
].copy()

X_validation = pd.get_dummies(
    X_validation,
    columns=['type'],
    dtype=int
)

X_validation = X_validation.reindex(
    columns=X_train.columns,
    fill_value=0
)

print("Validation shape:", X_validation.shape)
print("Missing values:", X_validation.isnull().sum().sum())

Validation shape: (1047331, 11)
Missing values: 0


In [269]:
validation_scores = iso_forest.decision_function(X_validation)

validation_df['anomaly_score'] = validation_scores

validation_df['anomaly_score'].describe()

count    1.047331e+06
mean     7.091714e-02
std      6.527985e-02
min     -2.891450e-01
25%      3.706298e-02
50%      9.676414e-02
75%      1.211847e-01
max      1.358705e-01
Name: anomaly_score, dtype: float64

In [270]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(-0.12, -0.221, -0.01)

validation_results = []

for threshold in thresholds:

    predictions = (
        validation_df['anomaly_score'] < threshold
    ).astype(int)

    precision = precision_score(
        validation_df['isFraud'],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        validation_df['isFraud'],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        validation_df['isFraud'],
        predictions,
        zero_division=0
    )

    flagged = predictions.sum()

    flagged_percentage = (
        flagged / len(predictions)
    ) * 100

    validation_results.append({
        'threshold': round(threshold, 2),
        'flagged': flagged,
        'flagged_percentage': flagged_percentage,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

validation_threshold_results = pd.DataFrame(
    validation_results
)

validation_threshold_results

,threshold,flagged,flagged_percentage,precision,recall,f1
0,-0.12,19903,1.900354,0.040195,0.382044,0.072737
1,-0.13,16770,1.601213,0.042993,0.344317,0.076442
2,-0.14,14138,1.349908,0.044561,0.300860,0.077624
3,-0.15,12192,1.164102,0.046178,0.268863,0.078818
4,-0.16,9708,0.926928,0.051813,0.240210,0.085240
5,-0.17,6888,0.657672,0.066202,0.217765,0.101536
6,-0.18,4326,0.413050,0.097550,0.201528,0.131464
7,-0.19,2519,0.240516,0.150457,0.180993,0.164318
8,-0.20,1446,0.138065,0.201245,0.138968,0.164407
9,-0.21,871,0.083164,0.251435,0.104585,0.147723


In [271]:
best_row = validation_threshold_results.loc[
    validation_threshold_results['f1'].idxmax()
]

best_row

threshold               -0.200000
flagged               1446.000000
flagged_percentage       0.138065
precision                0.201245
recall                   0.138968
f1                       0.164407
Name: 8, dtype: float64

In [272]:
X_final_test = final_test_df[
    numerical_features + ['type']
].copy()

X_final_test = pd.get_dummies(
    X_final_test,
    columns=['type'],
    dtype=int
)

X_final_test = X_final_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print("Final test shape:", X_final_test.shape)
print("Missing values:", X_final_test.isnull().sum().sum())

Final test shape: (201405, 11)
Missing values: 0


In [273]:
final_scores = iso_forest.decision_function(X_final_test)

final_test_df['anomaly_score'] = final_scores

In [274]:
selected_threshold = -0.18

final_test_df['predicted_anomaly'] = (
    final_test_df['anomaly_score'] < selected_threshold
).astype(int)

In [275]:
from sklearn.metrics import confusion_matrix, classification_report

final_cm = confusion_matrix(
    final_test_df['isFraud'],
    final_test_df['predicted_anomaly']
)

print("Confusion Matrix:")
print(final_cm)

Confusion Matrix:
[[198953    296]
 [  1733    423]]


In [276]:
print(
    classification_report(
        final_test_df['isFraud'],
        final_test_df['predicted_anomaly'],
        digits=4
    )
)

              precision    recall  f1-score   support

           0     0.9914    0.9985    0.9949    199249
           1     0.5883    0.1962    0.2943      2156

    accuracy                         0.9899    201405
   macro avg     0.7898    0.5974    0.6446    201405
weighted avg     0.9871    0.9899    0.9874    201405



In [277]:
final_flagged = final_test_df['predicted_anomaly'].sum()

final_flagged_percentage = (
    final_flagged / len(final_test_df)
) * 100

print("Total final-test transactions:", len(final_test_df))
print("Anomalies flagged:", final_flagged)
print("Percentage flagged:", final_flagged_percentage)

Total final-test transactions: 201405
Anomalies flagged: 719
Percentage flagged: 0.35699213028474963


# Model 2: Isolation Forest Improvement

## Analysis of Model 1 Errors

Model 1 achieved a precision of 58.16%, recall of 20.83%, and F1-score
of 30.67% on the final chronological test set using an anomaly threshold
of -0.18.

Although the model produced relatively few false-positive alerts, its
recall indicates that a substantial proportion of labelled fraudulent
transactions were not detected. Model 2 therefore investigates the
characteristics of detected and missed fraudulent transactions to guide
feature and model improvements.

## 6.1 Analysis of Model 1 Errors

Before developing the second model, the fraudulent transactions correctly
detected by Model 1 are compared with fraudulent transactions that were
missed. This analysis helps identify transaction characteristics that may
improve the anomaly detection model.

In [278]:
# Fraud transactions correctly detected by Model 1
detected_fraud = final_test_df[
    (final_test_df['isFraud'] == 1) &
    (final_test_df['predicted_anomaly'] == 1)
].copy()

# Fraud transactions missed by Model 1
missed_fraud = final_test_df[
    (final_test_df['isFraud'] == 1) &
    (final_test_df['predicted_anomaly'] == 0)
].copy()

print("Detected fraud:", len(detected_fraud))
print("Missed fraud:", len(missed_fraud))

Detected fraud: 423
Missed fraud: 1733
